In [ ]:
from datasets import load_dataset
from tqdm.auto import tqdm
import os
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import AutoModelForSequenceClassification
from transformers import pipeline
import copy
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
# from rich import print
import numpy as np
import pandas as pd
import random
# from parlai.utils.safety import OffensiveLanguageClassifier
from rouge_score import rouge_scorer
import evaluate

import warnings
warnings.filterwarnings('ignore')

# Data

In [4]:
df = pd.read_csv("paradetox.tsv", sep='\t', header=0)
df.drop(['neutral2', 'neutral3'], axis=1, inplace=True)

In [5]:
df.head()

,toxic,neutral1
0,he had steel balls too !,he was brave too!
1,"dude should have been taken to api , he would ...",It would have been good if he went to api. He ...
2,"im not gonna sell the fucking picture , i just...","I'm not gonna sell the picture, i just want to..."
3,the garbage that is being created by cnn and o...,the news that is being created by cnn and othe...
4,the reason they dont exist is because neither ...,The reason they don't exist is because neither...


In [6]:
def sort_two_arrays(arr1, arr2):
    """Sorts two arrays based on the order of the first array.

    Args:
    arr1: The array to sort by.
    arr2: The array to sort along with the first array.

    Returns:
    A tuple containing the sorted arr1 and arr2.
    """

    zipped_pairs = zip(arr1, arr2)
    sorted_pairs = sorted(zipped_pairs)

    sorted_arr1 = [x for x, _ in sorted_pairs]
    sorted_arr2 = [y for _, y in sorted_pairs]

    return sorted_arr1, sorted_arr2

In [8]:
prompts = []
toxic = []

for i, row in tqdm(df.iterrows(), total=df.shape[0]):
    prompts.append(row['toxic'])
    toxic.append(1)
    prompts.append(row['neutral1'])
    toxic.append(0)
        

prompts = np.array(prompts)
toxic = np.array(toxic)

keep_idx = random.sample(list(range(len(prompts))), 600)
prompts = prompts[keep_idx]
toxic = toxic[keep_idx]

toxic, prompts = sort_two_arrays(toxic, prompts)

100%|██████████| 11927/11927 [00:00<00:00, 67416.49it/s]


# Model

In [9]:
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = '0'

device = "cuda" if torch.cuda.is_available() else "cpu"

1


In [10]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf", torch_dtype=torch.bfloat16).to(device)

model = model.eval()

Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.27s/it]


# Hidden states

In [11]:
activations = {}
def get_activation(name):
    def hook(model, input, output):
        if (name in activations):
            activations[name].append(output[0])
        else:
            activations[name] = [output[0]]
    return hook


hooks = []
for i, layer in enumerate(model.model.layers):
    hooks.append(layer.self_attn.register_forward_hook(get_activation(i)))

In [12]:
hidden_states = []
activations = {}

pbar = tqdm(enumerate(prompts), total=len(prompts))
for i, p in pbar:

    pbar.set_description('cur prompt length (chars): %d' %(len(p)))
    
    with torch.no_grad():
        inputs = tokenizer(p, return_tensors="pt").input_ids
        outputs = model.forward(inputs.to(device))
        
    for k in activations.keys():
        activations[k] = torch.cat(activations[k], dim=1).detach().cpu()
        activations[k] =  torch.mean(activations[k], dim=-2) 


    hidden_states.append(copy.deepcopy(activations))
    activations = {}

cur prompt length (chars): 48: 100%|██████████| 600/600 [00:18<00:00, 32.93it/s] 


## Attractors

In [14]:
target_layers = [i for i in range(len(model.model.layers))]
all_prompts_attractors = hidden_states

In [15]:
attractors = {l: {} for l in [0, 1]}
for target_layer in target_layers:
    for l in [0, 1]:

        cur_attractors = []
        for i in range(len(all_prompts_attractors)):
            if (toxic[i] != l):
                continue
            cur_attractors.append(all_prompts_attractors[i][target_layer])

        cur_attractors = torch.stack(cur_attractors, dim=0)

        attractors[l][target_layer] = torch.mean(cur_attractors, dim=0)

# Detoxify

In [16]:
for hook in hooks:
    hook.remove()

In [17]:
target_style = 1
strength = -6
target_layer = 16


def change_behavior(target_layer):
    def hook(model, input, output):
        output = (output[0] + strength*attractors[target_style][target_layer].to(device), *output[1:])
        return output
    return hook


hook = model.model.layers[target_layer].self_attn.register_forward_hook(change_behavior(target_layer))

In [21]:
sentences = ['trumpstweets should be hysterical since the narcissistic buffoon has no sense of humor .',
             "sup gurl it 's been shit haven 't done a n y t h i n g",
             'ok - i can second this , that nicaraguan coffee is fucking amazing .']


for sentence in sentences:
    
    prompt = f'Original: "{sentence}"; Paraphrased: "'
    strength = -10

    with torch.no_grad():
        inputs = tokenizer(prompt, return_tensors="pt").input_ids
        outputs = model.generate(inputs.to(device), max_new_tokens=100, do_sample=False, top_k=50, top_p=0.95, pad_token_id=tokenizer.eos_token_id, use_cache=True).detach().cpu()
    output_text = tokenizer.decode(outputs[0, inputs.shape[1]:]).replace("\n","").replace("{","").replace("}","").replace('"','').strip('".').replace(',,','').replace('original','').replace('Original','').split('rewritten')[0].split('revised')[0].replace('10','').split('.')[0]
    
    print ('-------------------------------------------------')
    print (sentence)
    print (output_text)
        

-------------------------------------------------
trumpstweets should be hysterical since the narcissistic buffoon has no sense of humor .
Trump's tweets should be fun for the public since the president has no sense of humor
-------------------------------------------------
sup gurl it 's been shit haven 't done a n y t h i n g
sup gurl it's been a while, I've been busy
-------------------------------------------------
ok - i can second this , that nicaraguan coffee is fucking amazing .
I can confirm that the coffee from Nicaragua is excellent
